# Flux Balance, Flux Variability Analysis, and Flux Sampling
# Adopted from this tutorial in CobraPpy
# source (https://cobrapy.readthedocs.io/en/latest/simulating.html#Running-FVA)

In [ ]:
# reading the input files of the iMAT models and set the path to outputs
input_path = '/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/iMAT_models'
FBA_output = '/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/FBA_output'
Sampling_output = '/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/Flux_sampling'
FVA_output = '/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/FVA_output'


In [2]:
# note: this is running on a python virtual environment of version 3.9.6 set on terminal and activated via conda to be used as a kernel here 
# import the necessary packages 
import os
import openpyxl
import cobra
from cobra.flux_analysis import flux_variability_analysis # for flux variability analysis 
from cobra.flux_analysis import pfba # for parsimonous FBA
import pandas as pd
from cobra.io import load_matlab_model
from cobra.sampling import OptGPSampler # for Flux sampling 

# Flux Balance Analysis FBA

In [ ]:
# make a loop to read the models and run the FBA and then store the information in xlsx format
for filename in os.listdir(input_path):
    if filename.endswith('.mat'):
        # Create the full file path
        file_path = os.path.join(input_path, filename)
        
        # Load the MATLAB model
        print(f"Loading model: {filename}")
        model = load_matlab_model(file_path)

        # Run FBA
        solution = model.optimize() 
        fluxes = solution.fluxes

        # collect the information of the reaction FBA results and the other info
        flux_data = []
        for rxn in model.reactions:
            flux = fluxes[rxn.id]

            flux_data.append({
                'Reaction ID': rxn.id,
                'Reaction Name': rxn.name,
                'Flux': flux,
                'Subsystem': rxn.subsystem
            })  
        # Convert to DataFrames
        fluxes_df = pd.DataFrame(flux_data)

        # Save into one Excel file with three sheets: Objective, Uptake, Secretion
        excel_filename = filename.replace('.mat', '_FBA.xlsx') 
        excel_file = os.path.join(FBA_output, excel_filename)
        
        with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
            # Write Objective Value
            pd.DataFrame({'Objective Value': [solution.objective_value]}).to_excel(writer, sheet_name="Objective", index=False)

            # Write reaction fluxes
            fluxes_df.to_excel(writer, sheet_name="fluxes", index=False)

## Flux Sampling

In [ ]:
for i, filename in enumerate(os.listdir(input_path)):
    if filename.endswith('.mat'): 
        print(f"Processing file {i + 1}/{len(os.listdir(input_path))}: {filename}")
        
        file_path = os.path.join(input_path, filename)
        
        try:
            # Load the iMAT model
            print(f"Loading model: {filename}")
            model = load_matlab_model(file_path)
            model.solver = 'gurobi'
            
            # Run FBA to check solution feasibility
            solution = model.optimize()
            if solution.status != 'optimal':
                print(f"Model {filename} is not optimal. Skipping...")
                continue
            
            # Perform flux sampling
            print(f"Sampling the model: {filename}")
            optgp = OptGPSampler(model, processes=4)
            s = optgp.sample(1000) # 1000 iterations per reaction
            
            # Convert sampling results to DataFrame
            df = pd.DataFrame(s)
            csv_filename = os.path.splitext(filename)[0] + '_sampling.csv'
            csv_file_path = os.path.join(Sampling_output, csv_filename)
            df.to_csv(csv_file_path, index=False)
            print(f"Sampling results saved to: {csv_file_path}")
        
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

In [6]:
# flux sampling for one model at a time
# Set Gurobi as the solver
cobra.Configuration().solver = "gurobi"

# Confirm the solver
print("Selected Solver:", cobra.Configuration().solver)

# Load your model
model = cobra.io.load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/iMAT_models/iMAT_CP2-APPPS1-4.mat')

# Test optimization
solution = model.optimize()
print("Objective Value:", solution.objective_value)

Selected Solver: <module 'optlang.gurobi_interface' from '/opt/anaconda3/envs/python396_env/lib/python3.9/site-packages/optlang/gurobi_interface.py'>
Objective Value: 851.4420097101995


In [7]:
# run Flux Sampling
optgp = OptGPSampler(model, processes=4)
s = optgp.sample(1000)

Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp5v3ali6c.lp
Reading time = 0.01 seconds
: 2635 rows, 8242 columns, 34652 nonzeros


In [8]:
df = pd.DataFrame(s)
csv_filename = 'iMAT_CP2-APPPS1-4_sampling.csv'
csv_file_path = os.path.join(Sampling_output, csv_filename)
df.to_csv(csv_file_path, index=False)

# Flux Variability Analysis 

In [ ]:
# make a loop that read through each model in the file and then run FVA
# reults will include the reaction ID, its associated subsystem and both mic and max fluxes
for filename in os.listdir(input_path):
    if filename.endswith('.mat'): 
        # Create the full file path
        file_path = os.path.join(input_path, filename)
        
        # Load the MATLAB model
        print(f"Loading model: {filename}")
        model = load_matlab_model(file_path)

        print(f"Running FVA for the model: {filename}")

        # Run FVA for all reactions
        FVA = flux_variability_analysis(model, fraction_of_optimum=1.0)

        # Extract reaction information for only the lipid/energy reactions
        reaction_ids = []
        subsystems = []
        reaction_names = []
        genes = []

        for rxn in model.reactions:
            reaction_ids.append(rxn.id)
            subsystems.append(rxn.subsystem)
            reaction_names.append(rxn.name)
            genes.append(";".join([g.id for g in rxn.genes]))

        # Create DataFrame
        df = pd.DataFrame(FVA)
        
        # Add reaction metadata
        df['reaction_id'] = reaction_ids
        df['reaction_name'] = reaction_names
        df['subsystem'] = subsystems
        df['genes'] = genes
        
        # Reorder columns
        df = df[['reaction_id', 'reaction_name', 'subsystem', 'genes', 'minimum', 'maximum']]

        # Save as CSV
        csv_filename = os.path.splitext(filename)[0] + '_FVA.csv'
        csv_file_path = os.path.join(FVA_output, csv_filename)
        df.to_csv(csv_file_path, index=False)

        print(f"Saved FVA results to {csv_file_path}")